# Minimal demand system notebook: Senegal food demand

This notebook does the **simplest possible version** of the demand-system part of the project.

Goal: estimate a small system of food demands and show how food budget shares vary with **household need**.

What we will do:
1. Load the Senegal household data.
2. Build a simple **adult-equivalent need** measure from the RDI sheet.
3. Collapse many foods into **4 broad groups**.
4. Estimate one demand equation for each group using food budget shares.
5. Interpret the signs of the coefficients.

Model used:

\[
\text{share}_{ig} = \alpha_g + \beta_g \log\left(\frac{\text{total food exp}}{\text{adult equiv}}\right) + \gamma_g \log(\text{adult equiv}) + \text{region FE} + \varepsilon_{ig}
\]

Interpretation:
- If **β > 0**, the budget share for that food group rises as food spending per adult-equivalent rises.
- If **γ > 0**, larger-need households spend a larger share on that group, holding food spending per adult-equivalent fixed.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from pathlib import Path

if Path("Senegal Data.xlsx").exists():
    DATA_FILE = Path("Senegal Data.xlsx")
else:
    DATA_FILE = Path("/mnt/data/Senegal Data.xlsx")

DATA_FILE

## 1. Load the sheets we need

We only use three sheets:
- **Household Characteristics** for household composition,
- **RDI** for calorie requirements by age and sex,
- **Pivot Food Expenditures (2019)** for household food spending in 2019.


In [ ]:
hh = pd.read_excel(DATA_FILE, sheet_name="Household Characteristics")
rdi = pd.read_excel(DATA_FILE, sheet_name="RDI")
exp = pd.read_excel(DATA_FILE, sheet_name="Pivot Food Expenditures (2019)")

# Keep 2019 households only
hh = hh[hh["t"] == 2019].copy()

print("Households:", hh.shape[0])
print("Food expenditure rows:", exp.shape[0])
print("RDI rows:", rdi.shape[0])

## 2. Build a household-need measure

To keep things simple, I measure household need using **adult-equivalent calories**.

Steps:
- take the **Energy** row from the RDI sheet,
- multiply each household's counts in each age/sex bin by the corresponding calorie requirement,
- divide by the calorie requirement for a male age 19-30.

So:
- **adult_equiv = 1** means the household has the calorie need of one prime-age adult male,
- **adult_equiv = 2** means double that need, and so on.


In [ ]:
energy = rdi.loc[rdi["n"] == "Energy"].iloc[0]

need_map = {
    "Females 00-03": "F 00-03",
    "Males 00-03": "M 00-03",
    "Females 04-08": "F 04-08",
    "Males 04-08": "M 04-08",
    "Females 09-13": "F 09-13",
    "Males 09-13": "M 09-13",
    "Females 14-18": "F 14-18",
    "Males 14-18": "M 14-18",
    "Females 19-30": "F 19-30",
    "Males 19-30": "M 19-30",
    "Females 31-50": "F 31-50",
    "Males 31-50": "M 31-50",
    "Females 51+": "F 51+",
    "Males 51+": "M 51+",
}

hh["need_kcal"] = 0.0
for hh_col, rdi_col in need_map.items():
    hh["need_kcal"] += hh[hh_col].fillna(0) * float(energy[rdi_col])

# Normalize by energy need of a male age 19-30
hh["adult_equiv"] = hh["need_kcal"] / float(energy["M 19-30"])

hh[["i", "adult_equiv"]].head()

## 3. Collapse foods into a few broad groups

The original data has many food items. To keep the notebook easy to explain, I collapse them into **4 big groups**:
- **Staples and legumes**
- **Animal foods**
- **Fruits and vegetables**
- **Other foods**

This is not perfect, but it is enough for a simple class project demand system.


In [ ]:
food_cols = [c for c in exp.columns if c not in ["i", "m"]]

# Merge household need into the expenditure data
food = exp.merge(hh[["i", "adult_equiv"]], on="i", how="inner").copy()
food["region"] = food["m"]
food["total_food_exp"] = food[food_cols].sum(axis=1)

def classify_food(name):
    s = name.lower()

    if any(k in s for k in [
        "rice", "millet", "sorghum", "maize", "corn", "bread", "pasta",
        "flour", "semolina", "cassava", "yam", "potato", "sweet potato",
        "plantain", "fonio", "bean", "cowpea", "lentil", "pea", "groundnut",
        "peanut", "nut"
    ]):
        return "Staples and legumes"

    if any(k in s for k in [
        "beef", "meat", "fish", "chicken", "mutton", "goat", "pork",
        "shrimp", "shell", "oyster", "egg", "milk", "cheese", "yogurt", "curdled"
    ]):
        return "Animal foods"

    if any(k in s for k in [
        "leaf", "cabbage", "carrot", "tomato", "onion", "okra", "eggplant",
        "pepper", "spinach", "lettuce", "fruit", "banana", "orange", "mango",
        "papaya", "guava", "lemon", "lime", "melon", "apple", "avocado",
        "vegetable"
    ]):
        return "Fruits and vegetables"

    return "Other foods"

food_group_map = {col: classify_food(col) for col in food_cols}

# Sum expenditures within each broad group
for group in sorted(set(food_group_map.values())):
    group_cols = [c for c in food_cols if food_group_map[c] == group]
    food[group] = food[group_cols].sum(axis=1)

broad_groups = ["Staples and legumes", "Animal foods", "Fruits and vegetables", "Other foods"]
for g in broad_groups:
    food[f"share_{g}"] = food[g] / food["total_food_exp"]

food[["i", "adult_equiv", "total_food_exp"] + broad_groups].head()

## 4. Create the regression variables

We use:
- **log_x_per_ae** = log(total food expenditure / adult_equiv)
- **log_ae** = log(adult_equiv)

Then for each food group we estimate one equation with **region fixed effects**.


In [ ]:
food = food[(food["total_food_exp"] > 0) & (food["adult_equiv"] > 0)].copy()

food["log_x_per_ae"] = np.log(food["total_food_exp"] / food["adult_equiv"])
food["log_ae"] = np.log(food["adult_equiv"])

food[["log_x_per_ae", "log_ae"]].describe().T

## 5. Estimate the simple demand system

Each equation is estimated separately by OLS. To keep the code simple, region fixed effects are added with dummy variables.


In [ ]:
X = pd.DataFrame({
    "const": 1.0,
    "log_x_per_ae": food["log_x_per_ae"],
    "log_ae": food["log_ae"],
})

region_dummies = pd.get_dummies(food["region"], prefix="region", drop_first=True, dtype=float)
X = pd.concat([X, region_dummies], axis=1)

results = []
models = {}

for g in broad_groups:
    y = food[f"share_{g}"]
    model = sm.OLS(y, X).fit()
    models[g] = model
    results.append({
        "food_group": g,
        "beta_log_x_per_ae": model.params["log_x_per_ae"],
        "p_beta": model.pvalues["log_x_per_ae"],
        "gamma_log_ae": model.params["log_ae"],
        "p_gamma": model.pvalues["log_ae"],
        "R2": model.rsquared,
    })

results_df = pd.DataFrame(results)
results_df

## 6. Read the coefficients

How to read the table:
- **beta_log_x_per_ae**: how the budget share changes when food expenditure per adult-equivalent rises.
- **gamma_log_ae**: how the budget share changes for larger-need households, holding expenditure per adult-equivalent fixed.

A positive coefficient means the share rises; a negative coefficient means the share falls.


In [ ]:
def sign_text(x):
    if x > 0:
        return "positive"
    elif x < 0:
        return "negative"
    else:
        return "zero"

for _, row in results_df.iterrows():
    print(f"{row['food_group']}: beta is {sign_text(row['beta_log_x_per_ae'])}, gamma is {sign_text(row['gamma_log_ae'])}")

## 7. Make one simple graph

This graph shows the estimated **beta** and **gamma** coefficients for each food group.

- The first bar is the effect of higher food spending per adult-equivalent.
- The second bar is the effect of larger household need.


In [ ]:
plot_df = results_df.set_index("food_group")[["beta_log_x_per_ae", "gamma_log_ae"]]
plot_df.plot(kind="bar", figsize=(10, 5))
plt.axhline(0, linewidth=1)
plt.ylabel("Coefficient")
plt.title("Simple demand system coefficients")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 8. One paragraph you can say in class

We estimate a very simple demand system using Senegal 2019 household food expenditures. First, we convert household composition into an adult-equivalent measure using calorie requirements from the RDI sheet. Then we group foods into four broad categories and regress each category's food budget share on log food expenditure per adult-equivalent, log adult-equivalent household need, and region fixed effects. The coefficient on log expenditure per adult-equivalent shows how demand changes with resources, while the coefficient on log adult-equivalent shows how budget shares differ across households with greater nutritional need.
